# 02. Causal Mask와 Multi-Head Attention

## 학습 목표

- `d_model`을 head 축으로 나누고 다시 합친다.
- decoder의 causal mask가 미래 token 확률을 0으로 만드는지 검증한다.
- self-attention과 encoder-decoder cross-attention의 Q·K·V 출처를 구분한다.

In [ ]:
# numpy는 batched matrix multiplication과 reshape를 제공한다.
import numpy as np

# 결과를 재현할 수 있도록 seed가 고정된 generator를 만든다.
rng = np.random.default_rng(seed=2017)
# 작은 수를 읽기 쉽게 출력 자릿수를 제한한다.
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# 마지막 축에 안정적인 softmax를 적용한다.
def softmax(scores: np.ndarray) -> np.ndarray:
    # 행별 최댓값을 빼 exp overflow를 방지한다.
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    # 이동된 score를 지수값으로 변환한다.
    exp_scores = np.exp(shifted)
    # key 축 합으로 나눠 확률을 만든다.
    return exp_scores / exp_scores.sum(axis=-1, keepdims=True)

# [batch, sequence, d_model]을 [batch, head, sequence, d_head]로 바꾼다.
def split_heads(x: np.ndarray, num_heads: int) -> np.ndarray:
    # 입력의 세 축 크기를 이름 있는 변수로 분해한다.
    batch_size, sequence_length, d_model = x.shape
    # d_model이 head 수로 정확히 나뉘어야 정보 손실 없는 reshape가 가능하다.
    assert d_model % num_heads == 0
    # head 하나가 담당할 feature 차원을 계산한다.
    d_head = d_model // num_heads
    # 먼저 feature 축을 [head, d_head] 두 축으로 분리한다.
    reshaped = x.reshape(batch_size, sequence_length, num_heads, d_head)
    # attention 계산을 위해 head 축을 sequence 축 앞으로 이동한다.
    return reshaped.transpose(0, 2, 1, 3)

# split된 head를 다시 [batch, sequence, d_model]로 합친다.
def merge_heads(x: np.ndarray) -> np.ndarray:
    # head가 분리된 네 축의 크기를 읽는다.
    batch_size, num_heads, sequence_length, d_head = x.shape
    # sequence 축을 head 축 앞으로 되돌리고 contiguous copy를 만든다.
    reordered = x.transpose(0, 2, 1, 3).copy()
    # head와 d_head를 곱한 원래 d_model 축으로 합친다.
    return reordered.reshape(batch_size, sequence_length, num_heads * d_head)

In [ ]:
# head가 분리된 Q, K, V에 scaled dot-product attention을 적용한다.
def attention_by_head(
    q_heads: np.ndarray,
    k_heads: np.ndarray,
    v_heads: np.ndarray,
    allowed: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    # head당 key dimension을 scaling에 사용한다.
    d_head = float(q_heads.shape[-1])
    # K의 sequence와 feature 축을 바꿔 batched QK^T를 계산한다.
    scores = q_heads @ np.swapaxes(k_heads, -2, -1)
    # head dimension의 제곱근으로 score를 나눈다.
    scores = scores / np.sqrt(d_head)
    # mask가 있으면 허용되지 않은 연결을 큰 음수로 만든다.
    if allowed is not None:
        # allowed는 batch와 head 축으로 broadcast될 수 있다.
        scores = np.where(allowed, scores, -1.0e9)
    # 각 query가 key 위치에 부여할 확률을 계산한다.
    weights = softmax(scores)
    # 확률과 value를 곱해 head별 context를 만든다.
    return weights @ v_heads, weights

## Multi-head self-attention

교육용으로 `batch=1`, `sequence=4`, `d_model=8`, `head=2`를 사용한다. 실제 model에서는 Q, K, V projection이 학습된다.

In [ ]:
# token 네 개의 8차원 representation을 만든다.
x = rng.normal(size=(1, 4, 8))
# 세 projection matrix를 작은 random weight로 초기화한다.
w_q = rng.normal(scale=0.2, size=(8, 8))
# key projection은 query와 독립된 학습 parameter다.
w_k = rng.normal(scale=0.2, size=(8, 8))
# value projection도 별도 학습 parameter다.
w_v = rng.normal(scale=0.2, size=(8, 8))
# concat된 head를 섞을 output projection을 만든다.
w_o = rng.normal(scale=0.2, size=(8, 8))

# 같은 x에서 Q를 만들므로 self-attention이다.
q = x @ w_q
# 같은 x에서 K를 만든다.
k = x @ w_k
# 같은 x에서 V를 만든다.
v = x @ w_v
# d_model=8을 두 head의 d_head=4로 분할한다.
q_heads = split_heads(q, num_heads=2)
# K도 같은 head layout으로 분할한다.
k_heads = split_heads(k, num_heads=2)
# V도 같은 head layout으로 분할한다.
v_heads = split_heads(v, num_heads=2)
# mask 없는 head별 attention을 계산한다.
head_context, head_weights = attention_by_head(q_heads, k_heads, v_heads)
# head를 합친 뒤 output projection으로 subspace 정보를 섞는다.
multi_head_output = merge_heads(head_context) @ w_o

# split 결과는 [batch=1, head=2, sequence=4, d_head=4]여야 한다.
assert q_heads.shape == (1, 2, 4, 4)
# attention weight는 각 head마다 4×4 위치 관계를 가진다.
assert head_weights.shape == (1, 2, 4, 4)
# 최종 출력은 입력과 같은 d_model shape로 돌아와 residual addition이 가능하다.
assert multi_head_output.shape == x.shape
# 각 query 행의 key 확률 합이 1인지 확인한다.
assert np.allclose(head_weights.sum(axis=-1), 1.0)
# 관찰을 위해 첫 head의 attention matrix를 출력한다.
print('head 0 weights:\n', head_weights[0, 0])

## Decoder causal mask

아래 triangular mask는 query 위치 `i`가 key 위치 `j <= i`만 보게 한다. shape 앞의 두 `None` 축은 batch와 head에 broadcasting하기 위한 것이다.

In [ ]:
# sequence 길이 4의 아래쪽 triangular boolean matrix를 만든다.
causal = np.tril(np.ones((4, 4), dtype=bool))
# batch와 head 축을 추가해 [1, 1, query, key] mask로 만든다.
causal = causal[None, None, :, :]
# 같은 Q, K, V에 causal mask를 적용한다.
causal_context, causal_weights = attention_by_head(
    q_heads,
    k_heads,
    v_heads,
    allowed=causal,
)
# 첫 head의 masked attention을 출력한다.
print('causal head 0 weights:\n', causal_weights[0, 0])
# strict upper triangle은 미래 연결이므로 모두 0이어야 한다.
future_weights = np.triu(causal_weights[0, 0], k=1)
# 미래 token probability가 수치 오차 안에서 0인지 검증한다.
assert np.allclose(future_weights, 0.0)
# 현재와 과거 위치의 확률 합은 각 query마다 1이어야 한다.
assert np.allclose(causal_weights.sum(axis=-1), 1.0)
# mask가 있어도 context shape는 바뀌지 않는다.
assert causal_context.shape == q_heads.shape

## Encoder-decoder cross-attention

cross-attention에서는 Q가 decoder에서 오고 K·V가 encoder에서 온다. 따라서 query length와 key length가 달라도 된다.

In [ ]:
# decoder에는 출력 위치 세 개가 있다고 가정한다.
decoder_state = rng.normal(size=(1, 3, 8))
# encoder에는 입력 위치 다섯 개가 있다고 가정한다.
encoder_memory = rng.normal(size=(1, 5, 8))
# query만 decoder state에서 projection한다.
cross_q = split_heads(decoder_state @ w_q, num_heads=2)
# key는 encoder memory에서 projection한다.
cross_k = split_heads(encoder_memory @ w_k, num_heads=2)
# value도 encoder memory에서 projection한다.
cross_v = split_heads(encoder_memory @ w_v, num_heads=2)
# decoder query가 encoder의 모든 위치를 조회한다.
cross_context, cross_weights = attention_by_head(cross_q, cross_k, cross_v)
# weight의 마지막 두 축은 decoder length 3 × encoder length 5다.
assert cross_weights.shape == (1, 2, 3, 5)
# context는 decoder 위치마다 head별 vector 하나를 만든다.
assert cross_context.shape == (1, 2, 3, 4)
# key 방향 확률 합은 모든 decoder 위치와 head에서 1이다.
assert np.allclose(cross_weights.sum(axis=-1), 1.0)
# Q와 K length가 다른 결과 shape를 출력한다.
print('cross-attention weights shape:', cross_weights.shape)

## 통과 기준

- split과 merge 뒤 `d_model`이 보존된다.
- causal attention의 upper triangle이 0이다.
- cross-attention weight shape가 `decoder_length × encoder_length`임을 설명할 수 있다.
- 실제 제품 code에서는 padding mask와 causal mask를 논리 AND로 합치고 fully-masked row의 NaN 처리도 검증한다.